# NeuroGuard — Phase 5: final held-out evaluation
This project is a prototype and is not medical advice.

User-verified Phase 4 training and validation are complete. This notebook loads
the original models read-only. **Do not use Run all. Stop after section 8 for review.**
No retraining, threshold search, new split or postprocessing. All outputs stay
private in Drive. The student reviews preflight and explicitly approves inference.
Prepared with OpenAI Codex assistance; genuine test results are not pre-filled.
See `docs/phase5-evaluation.md` for fixed methods, recovery and interpretation.


## 1. Verify browser Colab T4
Runtime → Change runtime type → T4 GPU. Preserve installed CUDA PyTorch.
Use one runtime only. No VS Code extension or Drive API setup is needed.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, re
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
from google.colab import drive
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available() or 'T4' not in torch.cuda.get_device_name(0):
    raise RuntimeError('Use the original T4 CUDA hardware before continuing')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Enter the approved full evaluation source commit
Source must first be reviewed and made available by a separately approved push.
Opening the notebook does not clone its modules. No GitHub token is required.


In [ ]:
REVISION = input('Full reviewed Phase 5 Git commit SHA: ').strip().lower()
if not re.fullmatch(r'[0-9a-f]{40}', REVISION):
    raise ValueError('Full 40-character Git SHA required')
REPO = Path('/content/NeuroGuard')
REPOSITORY = 'https://github.com/hrb31/NeuroGuard.git'


## 3. Clone and verify public source
An existing edited/different checkout stops. Use a fresh runtime after preserving outputs.


In [ ]:
env = os.environ.copy()
env['GIT_TERMINAL_PROMPT'] = '0'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--quiet', '--no-checkout', REPOSITORY, str(REPO)], env=env, check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--quiet', '--detach', REVISION], env=env, check=True)
actual = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if actual != REVISION or subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
    raise RuntimeError('Existing checkout differs or is edited; use a clean runtime')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Evaluation source:', actual)


## 4. Install support packages and run synthetic checks
No dataset is accessed by these tests. Keep CUDA torch; pip check is strict.
If an imported supporting package changes, restart and repeat at the same SHA.


In [ ]:
torch_before = (str(torch.__version__), torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-training.txt'], check=True)
torch_after = json.loads(subprocess.check_output([sys.executable, '-c',
    'import torch,json; print(json.dumps([str(torch.__version__), torch.version.cuda]))'], text=True))
if tuple(torch_after) != torch_before:
    raise RuntimeError('PyTorch changed; stop for review')
test_run = subprocess.run([sys.executable, '-m', 'pytest', '-q'], text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(test_run.stdout)
test_run.check_returncode()
pip_check = subprocess.run([sys.executable, '-m', 'pip', 'check'], text=True,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(pip_check.stdout)
pip_check.check_returncode()


## 5. Mount Drive; fix original read-only inputs and new private output root
Never select a new run ID to evade prior test results. This workflow does not
overwrite Phase 2/3/4. A prior started/failed evaluation stops further inference.


In [ ]:
drive.mount('/content/drive')
from datetime import datetime, timezone
from src.data import write_json, digest
from src.dataset import file_sha256, load_manifest
from src.training import seed_everything
from src.final_evaluation import (assert_unused, current_environment, preflight, run_final_evaluation)
PROJECT = Path('/content/drive/MyDrive/NeuroGuard')
MANIFEST_PATH = PROJECT/'dataset-v2-seed42'/'split_manifest.json'
RUNS = {'bce': PROJECT/'experiments'/'phase3-bce'/'bce-seed42-01',
        'tversky': PROJECT/'experiments'/'phase4-tversky'/'tversky-seed42-01'}
RESULTS_ROOT = PROJECT/'experiments'/'phase5-final-evaluation'
assert_unused(RESULTS_ROOT)
PREFLIGHT = RESULTS_ROOT/'preflight'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
PREFLIGHT.mkdir(parents=True, exist_ok=False)
(PREFLIGHT/'pytest.txt').write_text(test_run.stdout)
(PREFLIGHT/'pip_check.txt').write_text(pip_check.stdout)
seed_everything(42)
environment = current_environment()
write_json(PREFLIGHT/'environment.json', environment)
print('Results stay private at:', RESULTS_ROOT)


## 6. Verify original manifest and frozen plan
Read-only identity/count/cutoff checks. No assignment or quantile is generated.
The test patient IDs and raw records are not printed.


In [ ]:
protocol = json.loads((REPO/'configs/final_evaluation.json').read_text())
manifest = load_manifest(MANIFEST_PATH, protocol['manifest_sha256'])
for split, counts in protocol['cohorts'].items():
    if [manifest['summary'][split][k] for k in ('patients', 'slices', 'positive_slices')] != counts:
        raise ValueError('Original frozen cohort counts differ')
print('Manifest identity:', manifest['manifest_sha256'])
print('Training-derived size cutoffs:', manifest['size_cutoffs'])
print('Frozen cohort summary:', json.dumps(manifest['summary'], indent=2))
print('Operating points:', json.dumps(protocol['operating_points'], indent=2))
print('Bootstrap:', protocol['bootstrap'])
print('Example policy:', protocol['qualitative_policy'])


## 7. Reuse/download version 2 inside Colab; resolve canonical root
Review [dataset release](https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation),
[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) and upstream terms.
Use Kaggle's secret/hidden login only if required; never send tokens back.
The existing resolver compares duplicate directory inventories. No full repeat
dataset audit, new manifest, development inference or test preview is performed.


In [ ]:
from src.acquisition import get_metadata, download_dataset, resolve_dataset_root
release = get_metadata()
if release['metadata']['licenseName'] != 'CC BY-NC-SA 4.0' or release['metadata']['currentVersionNumber'] != 2:
    raise RuntimeError('Release metadata changed; stop for review')
write_json(PREFLIGHT/'dataset_release.json', release)
terms_accepted = input('After reviewing dataset/upstream terms, type ACCEPT: ') == 'ACCEPT'
DOWNLOAD_ROOT = download_dataset(terms_accepted=terms_accepted)
DATA_ROOT, root_report = resolve_dataset_root(DOWNLOAD_ROOT)
write_json(PREFLIGHT/'root_resolution_PRIVATE.json', root_report)
print('Canonical version-2 root resolved; frozen file hashes will be verified next.')


## 8. Preflight only — STOP FOR REVIEW after this section
Confirm these are your own original trusted training directories, not downloaded
checkpoints from another source. Colocated checksums alone do not prove authenticity.
Verify no prior manual test evaluation took place outside this workflow.

Preflight compares source with both training commits; validates saved configs,
manifest, file bytes, architecture, best-history epoch, checkpoint hash/embedded
identity, original validation-selected thresholds and runtime/dependencies. It loads
the best models in eval mode and runs **synthetic zeros only**, with no gradients.
Missing or mismatched evidence stops; never regenerate metadata to bypass a check.
No test pixels are decoded or predicted before approval.


In [ ]:
trusted = input('Confirm original trusted checkpoints AND no previous test evaluation: type CONFIRM: ') == 'CONFIRM'
try:
    ticket, verified_models = preflight(REPO, MANIFEST_PATH, DATA_ROOT, RUNS, RESULTS_ROOT,
                                       environment, torch.device('cuda'), trusted_checkpoints=trusted)
except BaseException as error:
    write_json(PREFLIGHT/'preflight_failure.json', {'type': type(error).__name__, 'message': str(error),
                                                  'test_evaluated': False})
    raise
if ticket['source']['commit'] != REVISION:
    raise RuntimeError('Preflight source differs from entered revision')
write_json(PREFLIGHT/'preflight_report_PRIVATE.json', ticket)
PREFLIGHT_DIGEST = digest(ticket)
write_json(PREFLIGHT/'preflight_digest.json', {'sha256': PREFLIGHT_DIGEST})
print('PREFLIGHT DIGEST:', PREFLIGHT_DIGEST)
print('Test cohort:', ticket['test_cohort'])
for arm, evidence in ticket['checkpoints'].items():
    print(arm, 'source:', evidence['identity']['commit'], 'checkpoint SHA256:', evidence['hashes']['best.pt'])
    print('Epochs/best:', evidence['completed_epochs'], evidence['best_epoch'], 'selection:', evidence['selection'])
    print('Frozen config:', json.dumps(evidence['identity']['config'], indent=2))
    print('Supporting-package differences:', evidence['package_differences'])
print('Preflight passed. STOP here and review identities, methods, tests, counts, cutoffs and fallback status.')
del verified_models
torch.cuda.empty_cache()


## 9. Explicit final test-evaluation review gate
Only after reviewing section 8: change the flag to True and paste the printed
preflight digest below. This approves this exact frozen evaluation. Source publication
approval is separate. Leave these defaults unchanged until review is complete.
No edits to thresholds, models, size cutoffs or bootstrap/example policy.


In [ ]:
PREFLIGHT_REVIEWED = False
REVIEWED_PREFLIGHT_SHA256 = ''  # Paste the section 8 digest after review.


## 10. Evaluate once; save private reports
This cell prompts for `EVALUATE FROZEN TEST ONCE`, revalidates preflight and reloads
the original weights, then permanently records that evaluation started. No automatic
resume/repeat: preserve partial files and return any error for recovery review.
Inference uses the identical processed grid/metrics, no augmentation or cleanup.
One forward pass per model/batch is reused for all frozen thresholds.


In [ ]:
if not PREFLIGHT_REVIEWED or REVIEWED_PREFLIGHT_SHA256 != PREFLIGHT_DIGEST:
    raise RuntimeError('Review section 8 and complete the explicit section 9 gate first')
saved_ticket = json.loads((PREFLIGHT/'preflight_report_PRIVATE.json').read_text())
if digest(saved_ticket) != REVIEWED_PREFLIGHT_SHA256 or digest(ticket) != REVIEWED_PREFLIGHT_SHA256:
    raise RuntimeError('Reviewed preflight report changed')
approval = input('Type EVALUATE FROZEN TEST ONCE to approve final held-out inference: ')
OUTPUT, summary = run_final_evaluation(saved_ticket, approval=approval,
    reviewed_preflight_sha256=REVIEWED_PREFLIGHT_SHA256, device=torch.device('cuda'))
# Preserve the exact synthetic-check output alongside the final reproducibility record.
(OUTPUT/'pytest.txt').write_text(test_run.stdout)
(OUTPUT/'pip_check.txt').write_text(pip_check.stdout)
write_json(OUTPUT/'artifact_hashes.json', {p.relative_to(OUTPUT).as_posix(): file_sha256(p)
    for p in sorted(OUTPUT.rglob('*')) if p.is_file() and p.name != 'artifact_hashes.json'})
print('Saved private evaluation:', OUTPUT)


## 11. Review actual comparison and return evidence
Recall improvement and the FP guardrail are separate requirements. Recall gains
with failed FP guardrails mean partial support. Compare actual test findings with
validation honestly. No further model/threshold tuning from these results.
Neither model is clinically validated.

Return `review_summary.json`, the three CSV tables, bootstrap report, aggregate
figure, preflight check output/digest and pytest/pip output. Review the private FLAIR
figures for misses, boundaries and false positives. They use stable size examples
plus explicitly outcome-selected failures. Do not publish MRI/patient-level files
or weights; share only individually licensing/privacy-reviewed images.

Next phase remains Gradio and a separate polished judge-facing research notebook,
README/presentation, code simplification, full scientific/security/privacy and AI
attribution review, and final submission audit against the latest Veritas Briefs.
No Phase 6 implementation occurs here.


In [ ]:
print('Test cohort:', summary['cohort'])
print('Point | Dice | Recall | Precision | Small Dice | Small Recall | Empty FP | pos patients/slices | empty slices')
for name, r in summary['metrics'].items():
    p, s, e = r['positive_patient_macro'], r['size_groups']['small'], r['empty']
    values = [p['dice'], p['recall'], p['precision'], s['dice'], s['recall'], e['false_positive_rate']]
    print(name, '|', ' | '.join('NA' if v is None else f'{v:.5f}' for v in values),
          '|', p['patients'], '/', p['slices'], '|', e['slices'])
for point, comparison in summary['comparisons'].items():
    print(point, comparison['hypothesis_assessment'])
    print('Recall component:', comparison['recall_and_small_recall_improved'],
          'FP guardrail:', comparison['research_fp_guardrail_met'])
print('Private report:', OUTPUT/'review_summary.json')
from IPython.display import display, Image
display(Image(filename=str(OUTPUT/'aggregate_comparison.png')))
for path in sorted((OUTPUT/'figures_PRIVATE').glob('*.png')):
    display(Image(filename=str(path)))
